In [ ]:
from utils.preprocess import preliminary_preproc, separate_data_shift_categories
from utils.preprocess import preprocess_shift_categories_da, preprocess_shift_categories_sa, preprocess_data_shap_stability_analysis
from utils.models import RFModel, XGBModel, LGBMModel, GBTModel, LogRegModel
import shap
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
X, y, df = preliminary_preproc('./data/Telco_customer_churn.csv')

df_collection = separate_data_shift_categories(df)

In [ ]:
def train_and_eval_model(model_list, X_train, X_test, y_train, y_test, slice_name):

    trained_models = []

    for m in model_list:
        m.train(X_train, y_train)

        print(f"\nTesting {m.model_name} on {slice_name}:")
        m.evaluate(X_test, y_test)

        trained_models.append(m)

    return trained_models

In [ ]:
def shap_diagnostics(model, X_test, shap_dict):
    """SHAP Diagnostics"""

    model_name = model.__class__.__name__
    if model_name != "LogisticRegression":
        print(model_name)

        explainer = shap.TreeExplainer(model)
        shap_values = explainer(X_test)

        if model_name == "RandomForestClassifier":
            shap_values = shap_values[:, :, 1]

        shap_dict[model_name] = shap_values

        shap.plots.beeswarm(shap_values, show=False)

        plt.show()

In [ ]:
model_list = [RFModel(), XGBModel(), LGBMModel(), GBTModel(), LogRegModel()]

# 1. Distribution Analysis 

### 1.1 Tenure Cohort

In [ ]:
X_train_sm, X_test_sm, y_train_sm, y_test_sm = preprocess_shift_categories_da(df_collection['Tenure'],['Long'])
X_train_ml, X_test_ml, y_train_ml, y_test_ml = preprocess_shift_categories_da(df_collection['Tenure'],['Short'])

In [ ]:
trained_models_sm = train_and_eval_model(model_list, X_train_sm, X_test_sm, y_train_sm, y_test_sm, 'Long Tenure')

In [ ]:
trained_models_ml = train_and_eval_model(model_list, X_train_ml, X_test_ml, y_train_ml, y_test_ml, 'Short Tenure')

### SHAP Diagnostics

In [ ]:
shap_values_tenure_sm = {}
for m in trained_models_sm:
    shap_diagnostics(m.model, X_test_sm, shap_values_tenure_sm)

In [ ]:
shap_values_tenure_ml = {}
for m in trained_models_ml:
    shap_diagnostics(m.model, X_test_ml, shap_values_tenure_ml)

### 1.2 City

In [ ]:
cities = ['Top_3','Top_3_to_15', 'Top_15_and_above']

test_cat_1 = ['Top_15_and_above']
test_cat_2 = ['Top_3']
split_criteria = ['City']
blacklist = cities + ['Latitude','Longitude'] + split_criteria

X_train_15, X_test_15, y_train_15, y_test_15 = preprocess_shift_categories_da(df_collection['City'],test_cat_1, blacklist)
X_train_3, X_test_3, y_train_3, y_test_3 = preprocess_shift_categories_da(df_collection['City'],test_cat_2, blacklist)

In [ ]:
trained_models_t3 = train_and_eval_model(model_list, X_train_15, X_test_15, y_train_15, y_test_15, 'Top_3')

In [ ]:
trained_models_above_t15 = train_and_eval_model(model_list, X_train_3, X_test_3, y_train_3, y_test_3, 'Top_15_and_above')

### SHAP Diagnostics

In [ ]:
shap_values_city_t3 = {}
for m in trained_models_t3:
    shap_diagnostics(m.model, X_test_3, shap_values_city_t3)

In [ ]:
shap_values_city_above_t15 = {}
for m in trained_models_above_t15:
    shap_diagnostics(m.model, X_test_15, shap_values_city_above_t15)

### 1.3 Family Status

In [ ]:
test_cat_1 = ['Male_No_No', 'Female_No_No']
test_cat_2 = ['Female_No_Yes','Male_No_Yes']
test_cat_3 = ['Male_Yes_Yes', 'Female_Yes_Yes']

split_criteria = ['Gender','Partner','Dependents']

blacklist = split_criteria + list(df_collection['Status'].keys())

X_train_s1, X_test_s1, y_train_s1, y_test_s1 =\
             preprocess_shift_categories_da(df_collection['Status'],test_cat_1, blacklist)
X_train_s2, X_test_s2, y_train_s2, y_test_s2 =\
             preprocess_shift_categories_da(df_collection['Status'],test_cat_2, blacklist)
X_train_s3, X_test_s3, y_train_s3, y_test_s3 =\
             preprocess_shift_categories_da(df_collection['Status'],test_cat_3, blacklist)

In [ ]:
trained_models_s1 = train_and_eval_model(model_list, X_train_s1, X_test_s1, y_train_s1, y_test_s1, 'Single People without Children')

In [ ]:
trained_models_s2 = train_and_eval_model(model_list, X_train_s2, X_test_s2, y_train_s2, y_test_s2, 'Single Moms/Dads')

In [ ]:
trained_models_s3 = train_and_eval_model(model_list, X_train_s3, X_test_s3, y_train_s3, y_test_s3, 'Family People')

### SHAP Diagnostics

In [ ]:
shap_values_s1 = {}
for m in trained_models_s1:
    shap_diagnostics(m.model, X_test_s1, shap_values_s1)

In [ ]:
shap_values_s2 = {}
for m in trained_models_s2:
    shap_diagnostics(m.model, X_test_s2, shap_values_s2)

In [ ]:
shap_values_s3 = {}
for m in trained_models_s3:
    shap_diagnostics(m.model, X_test_s3, shap_values_s3)

# 2. Subpopulation Analysis 

### 2.1 Tenure Cohort

In [ ]:
X_train_sm, X_test_sm, y_train_sm, y_test_sm = preprocess_shift_categories_sa(X,y,df_collection['Tenure'],['Long'])
X_train_ml, X_test_ml, y_train_ml, y_test_ml = preprocess_shift_categories_sa(X,y,df_collection['Tenure'],['Short'])

In [ ]:
trained_models_sm = train_and_eval_model(model_list, X_train_sm, X_test_sm, y_train_sm, y_test_sm, 'Long Tenure')

In [ ]:
trained_models_ml = train_and_eval_model(model_list, X_train_ml, X_test_ml, y_train_ml, y_test_ml, 'Short Tenure')

### SHAP Diagnostics

In [ ]:
shap_values_tenure_sm_sub = {}
for m in trained_models_sm:
    shap_diagnostics(m.model, X_test_sm, shap_values_tenure_sm_sub)

In [ ]:
shap_values_tenure_ml_sub = {}
for m in trained_models_ml:
    shap_diagnostics(m.model, X_test_ml, shap_values_tenure_ml_sub)

### 2.2 City

In [ ]:
cities = ['Top_3','Top_3_to_15', 'Top_15_and_above']

test_cat_1 = ['Top_15_and_above']
test_cat_2 = ['Top_3']
split_criteria = ['City']
blacklist = cities + ['Latitude','Longitude'] + split_criteria

X_train_15, X_test_15, y_train_15, y_test_15 = preprocess_shift_categories_sa(X,y,df_collection['City'],test_cat_1, blacklist)
X_train_3, X_test_3, y_train_3, y_test_3 = preprocess_shift_categories_sa(X,y,df_collection['City'],test_cat_2, blacklist)

In [ ]:
trained_models_t3 = train_and_eval_model(model_list, X_train_15, X_test_15, y_train_15, y_test_15, 'Top_3')

In [ ]:
trained_models_above_t15 = train_and_eval_model(model_list, X_train_3, X_test_3, y_train_3, y_test_3, 'Top_15_and_above')

### SHAP Diagnostics

In [ ]:
shap_values_city_t3_sub = {}
for m in trained_models_t3:
    shap_diagnostics(m.model, X_test_3, shap_values_city_t3_sub)

In [ ]:
shap_values_city_above_t15_sub = {}
for m in trained_models_above_t15:
    shap_diagnostics(m.model, X_test_15, shap_values_city_above_t15_sub)

### 2.3 Family Status

In [ ]:
test_cat_1 = ['Male_No_No', 'Female_No_No']
test_cat_2 = ['Female_No_Yes','Male_No_Yes']
test_cat_3 = ['Male_Yes_Yes', 'Female_Yes_Yes']

split_criteria = ['Gender','Partner','Dependents']

blacklist = split_criteria + list(df_collection['Status'].keys())

X_train_s1, X_test_s1, y_train_s1, y_test_s1 =\
             preprocess_shift_categories_sa(X,y,df_collection['Status'],test_cat_1, blacklist)
X_train_s2, X_test_s2, y_train_s2, y_test_s2 =\
             preprocess_shift_categories_sa(X,y,df_collection['Status'],test_cat_2, blacklist)
X_train_s3, X_test_s3, y_train_s3, y_test_s3 =\
             preprocess_shift_categories_sa(X,y,df_collection['Status'],test_cat_3, blacklist)

In [ ]:
trained_models_s1 = train_and_eval_model(model_list, X_train_s1, X_test_s1, y_train_s1, y_test_s1, 'Single People without Children')

In [ ]:
trained_models_s2 = train_and_eval_model(model_list, X_train_s2, X_test_s2, y_train_s2, y_test_s2, 'Single Moms/Dads')

In [ ]:
trained_models_s3 = train_and_eval_model(model_list, X_train_s3, X_test_s3, y_train_s3, y_test_s3, 'Family People')

### SHAP Diagnostics

In [ ]:
shap_values_s1_sub = {}
for m in trained_models_s1:
    shap_diagnostics(m.model, X_test_s1, shap_values_s1_sub)

In [ ]:
shap_values_s2_sub = {}
for m in trained_models_s2:
    shap_diagnostics(m.model, X_test_s2, shap_values_s2_sub)

In [ ]:
shap_values_s3_sub = {}
for m in trained_models_s3:
    shap_diagnostics(m.model, X_test_s3, shap_values_s3_sub)

# 3. Advanced Analytics

### 3.1 SHAP Stability Across Subpopulations - Using Jensen-Shannon Divergence

In [ ]:
from scipy.spatial.distance import jensenshannon

def jsd_shap_feature(shap_values, mask_A, mask_B, feature_index, bins=50):
    shap_A = shap_values[mask_A, feature_index].values
    shap_B = shap_values[mask_B, feature_index].values

    # Convert to histograms (probability distributions)
    hist_A, bin_edges = np.histogram(shap_A, bins=bins, density=True)
    hist_B, _ = np.histogram(shap_B, bins=bin_edges, density=True)

    # Add small epsilon to avoid log(0)
    eps = 1e-10
    hist_A += eps
    hist_B += eps

    # Normalize
    hist_A /= hist_A.sum()
    hist_B /= hist_B.sum()

    return jensenshannon(hist_A, hist_B)**2

In [ ]:
def jsd_all_features(shap_values, mask_A, mask_B, feature_names):
    jsd_scores = {}

    for i, feature in enumerate(feature_names):
        jsd = jsd_shap_feature(shap_values, mask_A, mask_B, i)
        jsd_scores[feature] = jsd

    return jsd_scores

In [ ]:
# Visualization
import pandas as pd
def visualize_jsd_scores(jsd_scores, group='Tenure'):
    jsd_df = pd.DataFrame.from_dict(jsd_scores, orient="index", columns=["JSD"])
    jsd_df = jsd_df.sort_values("JSD", ascending=False)

    plt.figure(figsize=(10, 6))
    plt.barh(jsd_df.index, jsd_df["JSD"])
    plt.xlabel("Jensen-Shannon Divergence")
    plt.title(f"SHAP Distribution Shift Across {group} Groups")
    plt.gca().invert_yaxis()
    plt.show()

In [ ]:
# Visualization - heatmap
import seaborn as sns
def visualize_jsd_scores_heatmap(jsd_scores, group='Tenure'):
    jsd_df = pd.DataFrame(list(jsd_scores.items()), columns=['Feature', 'JSD'])

    # Reshape for heatmap (single column)
    jsd_heatmap = jsd_df.set_index('Feature')[['JSD']]

    plt.figure(figsize=(10, 6))
    sns.heatmap(jsd_heatmap, annot=True, fmt='.3f', cmap='YlOrRd', cbar_kws={'label': 'JSD Score'})
    plt.xlabel("Jensen-Shannon Divergence")
    plt.title(f"SHAP Distribution Shift Across {group} Groups")
    plt.gca().invert_yaxis()
    plt.show()

def visualize_jsd_scores_multi_group(jsd_scores_group, group='Tenure'):
    jsd_df = pd.DataFrame(jsd_scores_group)

    # sort by the average distance
    jsd_df['avg_shift'] = jsd_df.mean(axis=1)
    jsd_df = jsd_df.sort_values("avg_shift", ascending=True).drop(columns='avg_shift')

    plt.figure(figsize=(10, 6))
    jsd_df.plot(kind='barh', width=0.8, color=['#3474ad', '#eb8c34', '#47a047'])
    plt.xlabel("Jensen-Shannon Divergence")
    plt.title(f"SHAP Distribution Shift Across {group} Groups")
    plt.gca().invert_yaxis()
    plt.show()

def visualize_jsd_scores_radar_chart(jsd_scores_group, model_name, top_n=10):
    """radar chart for features with top 10 SHAP scores"""
    jsd_df = pd.DataFrame(jsd_scores_group)
    jsd_df['avg_shift'] = jsd_df.mean(axis=1)
    plot_df = jsd_df.sort_values('avg_shift', ascending=False).head(top_n).drop(columns='avg_shift')

    metrics = plot_df.index.tolist()
    num_vars = len(metrics)

    # Compute angle for each axis
    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

    # Define colors for your three categories
    colors = ['#3474ad', '#eb8c34', '#47a047']
    categories = plot_df.columns

    for i, cat in enumerate(categories):
        values = plot_df[cat].tolist()
        plot_values = values + [values[0]]
        plot_angles = angles + [angles[0]]

        
        ax.plot(plot_angles, plot_values, color=colors[i], linewidth=2, label=cat)
        ax.fill(plot_angles, plot_values, color=colors[i], alpha=0.15)

    # Fix axis labels and direction
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    plt.xticks(angles, metrics, size=10)

    plt.legend(loc='upper right', bbox_to_anchor=(1.2, 1.1))
    plt.title(f"Top Feature Distribution Shifts (JSD) - {model_name}", size=14, y=1.1)
    plt.show()


In [ ]:
# splitting data

X_train, X_test, y_train, y_test, mask_dict = preprocess_data_shap_stability_analysis(df, X, y)

In [ ]:
# training models

trained_models = train_and_eval_model(model_list, X_train, X_test, y_train, y_test, 'SHAP Stability')

In [ ]:
shap_scores_general = {}
for m in trained_models:
    shap_diagnostics(m.model, X_test, shap_scores_general)

3.1.1 Tenure Cohort

In [ ]:
jsd_scores_tenure_dict = {}
for model_name in shap_scores_general.keys():
    jsd_scores_tenure = jsd_all_features(shap_scores_general[model_name], mask_dict['X_mask_early_to_medium'], 
                            mask_dict['X_mask_medium_to_large'], X_test.columns)
    jsd_scores_tenure_dict[model_name] = dict(sorted(jsd_scores_tenure.items(), key=lambda x: x[1], reverse=True))

In [ ]:
for key in jsd_scores_tenure_dict.keys():
    visualize_jsd_scores(jsd_scores_tenure_dict[key])

3.1.2 City

In [ ]:
jsd_scores_city_dict = {}
for model_name in shap_scores_general.keys():
    jsd_scores_city = jsd_all_features(shap_scores_general[model_name], mask_dict['X_mask_city_top3'], 
                            mask_dict['X_mask_city_above_top15'], X_test.columns)
    jsd_scores_city_dict[model_name] = dict(sorted(jsd_scores_city.items(), key=lambda x: x[1], reverse=True))

In [ ]:
for key in jsd_scores_city_dict.keys():
    visualize_jsd_scores(jsd_scores_city_dict[key])

3.1.3 Family Status

In [ ]:
jsd_scores_status_dict = {}
for model_name in shap_scores_general.keys():
    jsd_scores_status_1 = jsd_all_features(shap_scores_general[model_name], mask_dict['Signle_without_children'], 
                            mask_dict['Signle_with_children'], X_test.columns)
    jsd_scores_status_2 = jsd_all_features(shap_scores_general[model_name], mask_dict['Signle_without_children'], 
                            mask_dict['Families_with_children'], X_test.columns)
    jsd_scores_status_3 = jsd_all_features(shap_scores_general[model_name], mask_dict['Signle_with_children'], 
                            mask_dict['Families_with_children'], X_test.columns)
    jsd_scores_status_dict[model_name] = {
        'Single with children vs. Single without children': dict(sorted(jsd_scores_status_1.items(), key=lambda x: x[1], reverse=True)),
        'Single with children vs. Families with children': dict(sorted(jsd_scores_status_2.items(), key=lambda x: x[1], reverse=True)),
        'Single with children vs. families with children': dict(sorted(jsd_scores_status_3.items(), key=lambda x: x[1], reverse=True))
    }

Multi-group barplot

In [ ]:
for key in jsd_scores_status_dict.keys():
    visualize_jsd_scores_multi_group(jsd_scores_status_dict[key])

Radar Chart

In [ ]:
for key in jsd_scores_status_dict.keys():
    visualize_jsd_scores_radar_chart(jsd_scores_status_dict[key], model_name=key)

### 3.2 Fairness / Bias Diagnostics

In [ ]:
from sklearn.metrics import confusion_matrix

def calculate_disparity_ratio(group_A, group_B):
    # We add a small epsilon to avoid division by zero
    return group_A.mean() / (group_B.mean()+1e-9)

def calculate_error_rate_ratio(group_A, group_B, y_test_A, y_test_B):
    preds_A = (group_A >= 0.5).astype(int)
    preds_B = (group_B >= 0.5).astype(int)

    tn_A, fp_A, fn_A, tp_A = confusion_matrix(y_test_A, preds_A).ravel()
    tn_B, fp_B, fn_B, tp_B = confusion_matrix(y_test_B, preds_B).ravel()

    fpr_A = fp_A / (fp_A + tn_A) if fp_A + tn_A > 0 else 0
    fpr_B = fp_B / (fp_B + tn_B) if fp_B + tn_B > 0 else 0

    return fpr_A / fpr_B

In [ ]:
# Visualization function

def visualize_ratios(ratios_arr, group_names, name="Disparity Ratio", model_name="RFModel"):
    fig, ax = plt.subplots(figsize=(10, 6))

    bars = ax.bar(group_names, ratios_arr, color='skyblue', edgecolor='navy', alpha=0.8)
    ax.tick_params(axis='x', labelrotation=45)

    ax.set_ylabel(name)
    ax.set_title(f'Fairness/Bias Diagnostic: {name}s by Group - {model_name}')

    ax.axhline(y=1.0, color='black', linestyle='-', linewidth=1.5, label='Perfect Parity')
    
    ax.axhline(y=0.8, color='red', linestyle='--', linewidth=1.2, label='0.8 Threshold (Lower)')
    ax.axhline(y=1.25, color='red', linestyle='--', linewidth=1.2, label='1.25 Threshold (Upper)')

    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom')
        
    plt.tight_layout()
    plt.show()

In [ ]:
def train_and_eval_model_fbd(model_list, X_train, X_test, y_train, y_test, mask_dict_aligned, ref_group = "X_mask_early_to_medium"):

    model_results = {}

    for m in model_list:
        m.train(X_train, y_train)

        _, _, y_probs = m.evaluate(X_test, y_test)

        y_probs = np.array(y_probs)

        ref_rates = y_probs[mask_dict_aligned[ref_group]] # masked
        y_test_ref = y_test[mask_dict_aligned[ref_group]] 

        model_name = m.__class__.__name__
        model_results[model_name] = {"disparity_ratios": {}, "error_rate_ratios": {}}

        for group_label, mask in mask_dict_aligned.items():
            group_rates = y_probs[mask]
            y_test_group = y_test[mask] 
            
            disparity_ratio = calculate_disparity_ratio(group_rates, ref_rates)
            error_rate_ratio = calculate_error_rate_ratio(group_rates, ref_rates, y_test_group, y_test_ref)
            model_results[model_name]["disparity_ratios"][group_label] = disparity_ratio
            model_results[model_name]["error_rate_ratios"][group_label] = error_rate_ratio

    return model_results


In [ ]:
X_train, X_test, y_train, y_test, mask_dict = preprocess_data_shap_stability_analysis(df, X, y) 
# use the same preprocessor, but no masks needed

3.2.1 Tenure Cohort

Training for Short to Medium Tenure

In [ ]:
model_results = train_and_eval_model_fbd(model_list, X_train, X_test, y_train, y_test, mask_dict, ref_group = "X_mask_early_to_medium")

Visualizing Disparity Ratios - Short to Medium Tenure

In [ ]:
for result in model_results.items():
    visualize_ratios(list(result[1]["disparity_ratios"].values()), list(result[1]["disparity_ratios"].keys()), model_name=result[0])

Visualizing Error Rates - Short to Medium Tenure

In [ ]:
for result in model_results.items():
    visualize_ratios(list(result[1]["error_rate_ratios"].values()), list(result[1]["error_rate_ratios"].keys()), name="error_rate_ratios", model_name=result[0])

Training for Mediumto Large Tenure

In [ ]:
model_results = train_and_eval_model_fbd(model_list, X_train, X_test, y_train, y_test, mask_dict, ref_group = "X_mask_medium_to_large")

Visualizing Disparity Ratios - Medium to Large Tenure

In [ ]:
for result in model_results.items():
    visualize_ratios(list(result[1]["disparity_ratios"].values()), list(result[1]["disparity_ratios"].keys()), model_name=result[0])

Visualizing Error Rates - Medium to Large Tenure

In [ ]:
for result in model_results.items():
    visualize_ratios(list(result[1]["error_rate_ratios"].values()), list(result[1]["error_rate_ratios"].keys()), name="error_rate_ratios", model_name=result[0])

3.2.2 City

Training for Top 3 Cities

In [ ]:
model_results = train_and_eval_model_fbd(model_list, X_train, X_test, y_train, y_test, mask_dict, ref_group = "X_mask_city_top3")

Visualizing Disparity Ratios - Top 3 Cities

In [ ]:
for result in model_results.items():
    visualize_ratios(list(result[1]["disparity_ratios"].values()), list(result[1]["disparity_ratios"].keys()), model_name=result[0])

Visualizing Error Rates - Top 3 Cities

In [ ]:
for result in model_results.items():
    visualize_ratios(list(result[1]["error_rate_ratios"].values()), list(result[1]["error_rate_ratios"].keys()), name="error_rate_ratios", model_name=result[0])

Training for Top 15 and Above Cities Cities 

In [ ]:
model_results = train_and_eval_model_fbd(model_list, X_train, X_test, y_train, y_test, mask_dict, ref_group = "X_mask_city_above_top15")

Visualizing Disparity Ratios - Top 15 and Above Cities Cities 

In [ ]:
for result in model_results.items():
    visualize_ratios(list(result[1]["disparity_ratios"].values()), list(result[1]["disparity_ratios"].keys()), model_name=result[0])

Visualizing Error Rates - Top 15 and Above Cities Cities 

In [ ]:
for result in model_results.items():
    visualize_ratios(list(result[1]["error_rate_ratios"].values()), list(result[1]["error_rate_ratios"].keys()), model_name=result[0])

3.2.3 Family Status

Training for Single without Children

In [ ]:
model_results = train_and_eval_model_fbd(model_list, X_train, X_test, y_train, y_test, mask_dict, ref_group = "Signle_without_children")

Visualizing Disparity Ratios - Single without Children

In [ ]:
for result in model_results.items():
    visualize_ratios(list(result[1]["disparity_ratios"].values()), list(result[1]["disparity_ratios"].keys()), model_name=result[0])

Visualizing Error Rates - Single without Children

In [ ]:
for result in model_results.items():
    visualize_ratios(list(result[1]["error_rate_ratios"].values()), list(result[1]["error_rate_ratios"].keys()), model_name=result[0])

Training for Single with Children

In [ ]:
model_results = train_and_eval_model_fbd(model_list, X_train, X_test, y_train, y_test, mask_dict, ref_group = "Signle_with_children")

Visualizing Disparity Ratios - Single with Children

In [ ]:
for result in model_results.items():
    visualize_ratios(list(result[1]["disparity_ratios"].values()), list(result[1]["error_rate_ratios"].keys()), model_name=result[0])

Visualizing Error Rates - Single with Children

In [ ]:
for result in model_results.items():
    visualize_ratios(list(result[1]["error_rate_ratios"].values()), list(result[1]["error_rate_ratios"].keys()), model_name=result[0])

Training for Families with Children

In [ ]:
model_results = train_and_eval_model_fbd(model_list, X_train, X_test, y_train, y_test, mask_dict, ref_group = "Families_with_children")

Visualizing Disparity Ratios - Families with Children

In [ ]:
for result in model_results.items():
    visualize_ratios(list(result[1]["disparity_ratios"].values()), list(result[1]["disparity_ratios"].keys()), model_name=result[0])

Visualizing Error Rates - Families with Children

In [ ]:
for result in model_results.items():
    visualize_ratios(list(result[1]["error_rate_ratios"].values()), list(result[1]["error_rate_ratios"].keys()), model_name=result[0])